#### CMSE 381 Final Project
### ✅ Group members: Evan Wood, Eric Grenadier
### ✅ Section_001
#### ✅ 4/14/2026

# Predicting Student Exam Scores: Identifying Key Performance Factors

## Background and Motivation

Our goal for this project is to look into understanding which student related factors are going to end up most strongly relating to the academic performance, meaning specifically exam scores. As students can be influenced by many variables such as study habits, attendance, family background, sleep hours, so its very important to determine which of these have the biggest impact or influence on a students score results to help improve students grades and learning.

In this project we are going to be looking to answer the question of **which student factors are most related to exam scores?** To do this we are going to need to analyze the dataset and apply regression models to help identify key variables and measure out their influence on student performance.

## Methodology

To be able to answer our question of what variables influence exam scores the most we first explored the dataset more to help understand the variables to identify and which features would be better to use to predict exam scores. Following this we then cleaned the data and handled any of the missing values and converted any categorical variables into numerical format so we can use them in our models. 

Following this we split the data into training and testing sets so we can evaluate the models performance. We use regression models including linear regression and random forest to predict exam scores based on input features. To improve our results we apply cross-validation techniques for parameter selection and feature selection to reduce overfitting and identify the most important predictors.

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor

### Data

For the dataset used for this project we used the "StudentPerformanceFactors" dataset, this gave us info on students academic performance along with various demographic behaviors. With all the info given we are going to be focusing on the target variable of "Exam_Score" this represents a students performance and will give us insight when comparing that target data point to other columns.

The dataset includes variables like Hours_Studied, Attendance, Sleep_Hours, and Previous_Scores, as well as categorical variables like Parental_Involvement, Motivation_Level, Family_Income, and Teacher_Quality. These are just some of the variables that each represent different aspects of a students academic habits, behaviors and background.

For this we chose to use all available variables as they can all provide a wide range of potential influence on a students performance. By looking at all variables allows us to analyse which factors are going to be most strongly related to the exam scores which will help us then expand and build more accurate predictive models.

In [18]:
df =pd.read_csv("StudentPerformanceFactors.csv")
print("shape of dataset:",df.shape )
print("\ncolumns:")
print(df.columns.tolist())

print("\nmissing values by column:")
print(df.isnull( ).sum() )

target=  "Exam_Score"
feature_cols= [ col for col in  df.columns if col !=target]

print("\ntarget variable:")
print(target)
print("\nfeature variables:")
print(feature_cols)
numeric_cols= df[feature_cols].select_dtypes( include=[ "int64", "float64" ] ).columns.tolist()
categorical_cols= df[feature_cols ].select_dtypes(include=["object", "str"] ).columns.tolist( )
print("\nnumeric columns:")
print(numeric_cols)

print("\ncategorical columns:")
print(categorical_cols)

shape of dataset: (6607, 20)

columns:
['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender', 'Exam_Score']

missing values by column:
Hours_Studied                  0
Attendance                     0
Parental_Involvement           0
Access_to_Resources            0
Extracurricular_Activities     0
Sleep_Hours                    0
Previous_Scores                0
Motivation_Level               0
Internet_Access                0
Tutoring_Sessions              0
Family_Income                  0
Teacher_Quality               78
School_Type                    0
Peer_Influence                 0
Physical_Activity              0
Learning_Disabilities          0
Parental_Educa

In [19]:
display(df.head(10))

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70
5,19,88,Medium,Medium,Yes,8,89,Medium,Yes,3,Medium,Medium,Public,Positive,3,No,Postgraduate,Near,Male,71
6,29,84,Medium,Low,Yes,7,68,Low,Yes,1,Low,Medium,Private,Neutral,2,No,High School,Moderate,Male,67
7,25,78,Low,High,Yes,6,50,Medium,Yes,1,High,High,Public,Negative,2,No,High School,Far,Male,66
8,17,94,Medium,High,No,6,80,High,Yes,0,Medium,Low,Private,Neutral,1,No,College,Near,Male,69
9,23,98,Medium,Medium,Yes,8,71,Medium,Yes,0,High,High,Public,Positive,5,No,High School,Moderate,Male,72


**Data Preprocessing**

Before building our models we need to handle some data quality issues. There are missing values in three categorical columns: Teacher_Quality (78 missing), Parental_Education_Level (90 missing), and Distance_from_Home (67 missing). Since these are categorical we fill them with the mode (most frequent value) for each column.

After handling missing values we one-hot encode all categorical features so they can be used in our regression models. Finally we split the data into 80% training and 20% testing sets.

In [20]:
# fill missing values with the mode for each column
for col in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    df[col] = df[col].fillna(df[col].mode()[0])

print("missing values after filling:")
print(df.isnull().sum().sum(), "total missing values")

missing values after filling:
0 total missing values


In [21]:
# one-hot encode categorical features
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("shape after encoding:", df_encoded.shape)
print("\nencoded columns:")
print(df_encoded.columns.tolist())

shape after encoding: (6607, 28)

encoded columns:
['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity', 'Exam_Score', 'Parental_Involvement_Low', 'Parental_Involvement_Medium', 'Access_to_Resources_Low', 'Access_to_Resources_Medium', 'Extracurricular_Activities_Yes', 'Motivation_Level_Low', 'Motivation_Level_Medium', 'Internet_Access_Yes', 'Family_Income_Low', 'Family_Income_Medium', 'Teacher_Quality_Low', 'Teacher_Quality_Medium', 'School_Type_Public', 'Peer_Influence_Neutral', 'Peer_Influence_Positive', 'Learning_Disabilities_Yes', 'Parental_Education_Level_High School', 'Parental_Education_Level_Postgraduate', 'Distance_from_Home_Moderate', 'Distance_from_Home_Near', 'Gender_Male']


In [22]:
# split into features and target, then train/test split
X = df_encoded.drop(columns=["Exam_Score"])
y = df_encoded["Exam_Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("training set:", X_train.shape)
print("testing set:", X_test.shape)

training set: (5285, 27)
testing set: (1322, 27)


### Models for regression

We use two regression models to predict exam scores:

**Linear Regression** — This is our baseline model. It fits a linear relationship between all input features and the target variable (Exam_Score). We chose this because it is interpretable and lets us directly read off which features have the largest effect through their coefficients. We evaluate using mean squared error (MSE) and R² score on a held-out test set.

**Random Forest Regression** — This is our second model. Unlike linear regression, Random Forest is an ensemble method that builds many decision trees and averages their predictions. It can capture non-linear relationships and interactions between features. We use the same MSE and R² metrics so we can directly compare performance against linear regression. Random Forest also provides feature importance scores which give us another way to rank which variables matter most.

In [23]:
# in principle, linear regression finds coefficients b such that:
#   Exam_Score = b0 + b1*Hours_Studied + b2*Attendance + ... + bn*Feature_n
# it minimizes the sum of squared residuals between predicted and actual scores

# random forest builds many decision trees on random subsets of the data
# and averages their predictions to reduce variance and overfitting

### Other methods used

To improve each of our regression models we apply two additional techniques:

**Lasso Regression with K-Fold Cross-Validation** — To improve on ordinary linear regression we apply Lasso (L1 regularization) which adds a penalty on the absolute size of coefficients. This serves two purposes: it uses 5-fold cross-validation to select the best regularization strength (alpha), and it performs automatic feature/subset selection by shrinking unimportant coefficients to exactly zero. This reduces the number of parameters in the model and helps prevent overfitting.

**Random Forest Hyperparameter Tuning with K-Fold Cross-Validation** — The default Random Forest uses arbitrary hyperparameters. We use GridSearchCV with 5-fold cross-validation to search over different values of n_estimators (number of trees), max_depth (maximum tree depth), and min_samples_split (minimum samples to split a node). This finds the combination of parameters that gives the best average cross-validated performance.

In [24]:
# Lasso works like linear regression but adds a penalty term:
#   minimize: sum((y - X*b)^2) + alpha * sum(|b|)
# the alpha parameter controls how much regularization is applied
# LassoCV tests many alpha values using cross-validation to find the best one

# GridSearchCV exhaustively tests every combination of hyperparameters
# and evaluates each using k-fold cross-validation to find the best set

## Results

Below we present the results of fitting each model on the training data and evaluating on the held-out test set. We compare performance across all four approaches: ordinary linear regression, random forest, Lasso with cross-validated alpha, and random forest with tuned hyperparameters.

### Regression results

We first fit ordinary linear regression as our baseline, then fit a random forest regressor. For each model we report MSE and R² on the test set, and examine which features are most important.

In [25]:
# fit linear regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# predict on test set
y_pred_lr = lr_model.predict(X_test)

# evaluate
lr_mse = mean_squared_error(y_test, y_pred_lr)
lr_r2 = r2_score(y_test, y_pred_lr)

print("Linear Regression Results:")
print(f"  Mean Squared Error: {lr_mse:.4f}")
print(f"  R² Score:           {lr_r2:.4f}")

Linear Regression Results:
  Mean Squared Error: 3.2560
  R² Score:           0.7696


In [26]:
# top 10 most influential features by absolute coefficient value
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr_model.coef_
})
coef_df["Abs_Coefficient"] = np.abs(coef_df["Coefficient"])
coef_df = coef_df.sort_values("Abs_Coefficient", ascending=False)

print("Top 10 features by coefficient magnitude:")
print(coef_df.head(10).to_string(index=False))

Top 10 features by coefficient magnitude:
                    Feature  Coefficient  Abs_Coefficient
    Access_to_Resources_Low    -2.101150         2.101150
   Parental_Involvement_Low    -2.001684         2.001684
          Family_Income_Low    -1.108291         1.108291
Parental_Involvement_Medium    -1.069448         1.069448
    Peer_Influence_Positive     1.051673         1.051673
       Motivation_Level_Low    -1.043881         1.043881
 Access_to_Resources_Medium    -1.031581         1.031581
        Teacher_Quality_Low    -1.015959         1.015959
        Internet_Access_Yes     0.958665         0.958665
  Learning_Disabilities_Yes    -0.857212         0.857212


The linear regression coefficients tell us how much the predicted exam score changes for a one-unit increase in each feature. The features with the largest absolute coefficients have the strongest linear relationship with exam scores. Next we fit a Random Forest to see if capturing non-linear patterns improves performance.

In [27]:
# fit random forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# predict on test set
y_pred_rf = rf_model.predict(X_test)

# evaluate
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print(f"  Mean Squared Error: {rf_mse:.4f}")
print(f"  R² Score:           {rf_r2:.4f}")

Random Forest Results:
  Mean Squared Error: 4.9872
  R² Score:           0.6472


In [28]:
# feature importance from random forest
rf_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 10 most important features (Random Forest):")
print(rf_importance.head(10).to_string(index=False))

Top 10 most important features (Random Forest):
                              Feature  Importance
                           Attendance    0.380896
                        Hours_Studied    0.242241
                      Previous_Scores    0.092234
                    Tutoring_Sessions    0.039065
                          Sleep_Hours    0.027021
                    Physical_Activity    0.026876
             Parental_Involvement_Low    0.020537
              Access_to_Resources_Low    0.017597
           Access_to_Resources_Medium    0.011180
Parental_Education_Level_Postgraduate    0.010913


The Random Forest feature importances tell us how much each feature contributes to reducing prediction error across all trees. We can compare these rankings against the linear regression coefficients to see which features are consistently important across both models. Next we apply our improvement methods to see if we can get better performance.

### Other results

Here we apply our improvement methods: Lasso with K-fold CV for feature selection on the linear model, and GridSearchCV for hyperparameter tuning on the Random Forest. We compare each improved model against its baseline version.

In [29]:
# fit Lasso with 5-fold CV to find best alpha
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train, y_train)

print(f"Best alpha selected by 5-fold CV: {lasso_cv.alpha_:.6f}")

# predict on test set
y_pred_lasso = lasso_cv.predict(X_test)

# evaluate
lasso_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)

print(f"\nLasso Regression Results:")
print(f"  Mean Squared Error: {lasso_mse:.4f}")
print(f"  R² Score:           {lasso_r2:.4f}")

print(f"\nComparison to Ordinary Linear Regression:")
print(f"  OLS MSE:   {lr_mse:.4f}  |  Lasso MSE:   {lasso_mse:.4f}")
print(f"  OLS R²:    {lr_r2:.4f}  |  Lasso R²:    {lasso_r2:.4f}")

Best alpha selected by 5-fold CV: 0.026155

Lasso Regression Results:
  Mean Squared Error: 3.3300
  R² Score:           0.7644

Comparison to Ordinary Linear Regression:
  OLS MSE:   3.2560  |  Lasso MSE:   3.3300
  OLS R²:    0.7696  |  Lasso R²:    0.7644


In [30]:
# show which features Lasso kept vs removed
lasso_coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lasso_cv.coef_
})

kept = lasso_coef_df[lasso_coef_df["Coefficient"] != 0].sort_values("Coefficient", key=abs, ascending=False)
dropped = lasso_coef_df[lasso_coef_df["Coefficient"] == 0]

print(f"Features kept by Lasso: {len(kept)} / {len(lasso_coef_df)}")
print(f"Features dropped (coefficient = 0): {len(dropped)}\n")

print("Kept features (sorted by importance):")
print(kept.to_string(index=False))

if len(dropped) > 0:
    print(f"\nDropped features:")
    print(dropped["Feature"].tolist())

Features kept by Lasso: 24 / 27
Features dropped (coefficient = 0): 3

Kept features (sorted by importance):
                              Feature  Coefficient
              Access_to_Resources_Low    -1.762820
             Parental_Involvement_Low    -1.691306
          Parental_Involvement_Medium    -0.832342
           Access_to_Resources_Medium    -0.786084
                    Family_Income_Low    -0.772670
              Peer_Influence_Positive     0.742189
                 Motivation_Level_Low    -0.675684
            Learning_Disabilities_Yes    -0.589267
                  Internet_Access_Yes     0.588563
                  Teacher_Quality_Low    -0.562874
              Distance_from_Home_Near     0.492821
                    Tutoring_Sessions     0.492058
       Extracurricular_Activities_Yes     0.479042
Parental_Education_Level_Postgraduate     0.403679
 Parental_Education_Level_High School    -0.383910
               Teacher_Quality_Medium    -0.346777
                        

The Lasso model used 5-fold cross-validation to select the best alpha and automatically dropped features that did not contribute meaningfully to predicting exam scores. This gives us a simpler, more interpretable model. Next we apply GridSearchCV to tune the Random Forest hyperparameters.

In [31]:
# tune random forest with GridSearchCV (5-fold CV)
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Best parameters from 5-fold CV:")
for param, val in grid_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"  Best CV MSE: {-grid_search.best_score_:.4f}")

Best parameters from 5-fold CV:
  max_depth: 20
  min_samples_split: 10
  n_estimators: 200
  Best CV MSE: 6.1765


In [32]:
# evaluate the tuned model on the test set
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test)

tuned_mse = mean_squared_error(y_test, y_pred_tuned)
tuned_r2 = r2_score(y_test, y_pred_tuned)

print("Tuned Random Forest Results:")
print(f"  Mean Squared Error: {tuned_mse:.4f}")
print(f"  R² Score:           {tuned_r2:.4f}")

print(f"\nComparison to Default Random Forest:")
print(f"  Default MSE: {rf_mse:.4f}  |  Tuned MSE: {tuned_mse:.4f}")
print(f"  Default R²:  {rf_r2:.4f}  |  Tuned R²:  {tuned_r2:.4f}")

Tuned Random Forest Results:
  Mean Squared Error: 4.7510
  R² Score:           0.6639

Comparison to Default Random Forest:
  Default MSE: 4.9872  |  Tuned MSE: 4.7510
  Default R²:  0.6472  |  Tuned R²:  0.6639


In [33]:
# summary comparison of all models
summary = pd.DataFrame({
    "Model": ["Linear Regression", "Lasso (CV-tuned)", "Random Forest (default)", "Random Forest (CV-tuned)"],
    "MSE": [lr_mse, lasso_mse, rf_mse, tuned_mse],
    "R²": [lr_r2, lasso_r2, rf_r2, tuned_r2]
}).sort_values("R²", ascending=False)

print("Model Comparison Summary:")
print(summary.to_string(index=False))

Model Comparison Summary:
                   Model      MSE       R²
       Linear Regression 3.256020 0.769650
        Lasso (CV-tuned) 3.329965 0.764418
Random Forest (CV-tuned) 4.751003 0.663885
 Random Forest (default) 4.987183 0.647177


## Discussion and Conclusion

### Discussion on the regression results

Linear regression achieved an R² of 0.770 and MSE of 3.256, meaning it explains about 77% of the variance in exam scores. The most influential features by coefficient magnitude were Access_to_Resources_Low (-2.10), Parental_Involvement_Low (-2.00), and Family_Income_Low (-1.11). These are all negative, meaning students with low access to resources score roughly 2.1 points lower on exams, students with low parental involvement score about 2.0 points lower, and students from low income families score about 1.1 points lower, holding all else constant. On the positive side, having positive peer influence adds about 1.05 points and having internet access adds about 0.96 points.

Random Forest achieved an R² of 0.647 and MSE of 4.987, which is notably worse than linear regression. This was somewhat surprising since Random Forest can capture non-linear relationships. The feature importance rankings from Random Forest tell a different story than the linear regression coefficients: Attendance (0.381) and Hours_Studied (0.242) dominate as the two most important features, together accounting for over 62% of the total importance. Previous_Scores (0.092) is a distant third. This suggests that the numeric study-habit features drive the most variation in the tree-based model, while the categorical features like parental involvement and resources showed up more strongly in the linear model's coefficients.

### Discussion on the other results

Lasso regression with 5-fold cross-validation selected an alpha of 0.026 and achieved an R² of 0.764 and MSE of 3.330. This is very close to ordinary linear regression (R² = 0.770) but with a simpler model. Lasso dropped 3 out of 27 features by setting their coefficients to zero: School_Type_Public, Distance_from_Home_Moderate, and Gender_Male. This tells us that whether a student attends a public or private school, their distance from home, and their gender do not meaningfully predict exam scores once the other variables are accounted for. The remaining 24 features were sufficient to achieve nearly the same performance, giving us a more parsimonious and interpretable model.

The tuned Random Forest with GridSearchCV selected the best hyperparameters as max_depth=20, min_samples_split=10, and n_estimators=200. This improved the default Random Forest from R² = 0.647 (MSE = 4.987) to R² = 0.664 (MSE = 4.751), a reduction in MSE of about 4.7%. The key change was constraining max_depth to 20 and requiring at least 10 samples to split a node, which reduced overfitting compared to the default unlimited depth. However, even with tuning the Random Forest still underperformed both linear models, suggesting that the relationship between these student factors and exam scores is largely linear in nature.

### Conclusion and future steps

To answer our research question of which student factors are most related to exam scores: the linear regression coefficients show that Access_to_Resources and Parental_Involvement have the strongest effects, with students in the "Low" category for either scoring roughly 2 points lower on exams. The Random Forest importance rankings highlight Attendance and Hours_Studied as the dominant predictors, together explaining over 62% of the model's predictive power. Across both models, the consistent finding is that study habits (hours studied, attendance), prior academic performance, and family/resource factors (parental involvement, access to resources, family income) are the strongest predictors of exam scores, while school type, distance from home, and gender have little to no effect.

Our best performing model was ordinary linear regression with an R² of 0.770, explaining 77% of variance in exam scores. The Lasso improvement showed that we can drop 3 features with minimal loss in accuracy (R² = 0.764), giving a simpler model. The Random Forest models performed worse overall but provided a useful complementary view of feature importance through their tree-based rankings.

For future steps, we could explore interaction terms between features (for example whether hours studied matters more for students with low parental involvement), try other regularization methods like Ridge or Elastic Net, or apply polynomial features to capture non-linear effects within the linear framework. We could also explore other ensemble methods like Gradient Boosting which may perform better than Random Forest on this dataset.

## Author Contribution

- **Evan Wood**: Data exploration, data loading, background and motivation writeup, methodology writeup, results analysis, discussion and conclusion.
- **Eric Grenadier**: Data preprocessing, linear regression, random forest, Lasso CV, GridSearchCV hyperparameter tuning.

## References

- "Student Performance Factors" dataset from Kaggle: https://www.kaggle.com/datasets/lainguyn123/student-performance-factors
- Scikit-learn documentation: https://scikit-learn.org/stable/